# Capítulo 5 — Incertidumbre y medida

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Por qué dos parámetros bien determinados pueden dar una predicción pésima?

Ajusta una exponencial a datos ruidosos, dibuja la elipse de covarianza de los
parámetros y muestra el efecto de ignorar su correlación al propagar.

La figura responde: ¿qué información pierdo si sólo guardo las barras de error
y no la matriz de covarianza?

Ejecutar:  python fig_covarianza.py

*(script original: `codigo/fig_covarianza.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(33)


def modelo(t, A, tau):
    return A * np.exp(-t / tau)


A_REAL, TAU_REAL, SIGMA = 100.0, 4.0, 4.0
t = np.linspace(0, 6, 14)
y = modelo(t, A_REAL, TAU_REAL) + r.normal(0, SIGMA, t.size)

popt, pcov = curve_fit(modelo, t, y, p0=[80, 3], sigma=np.full(t.size, SIGMA),
                       absolute_sigma=True)
sA, stau = np.sqrt(np.diag(pcov))
rho = pcov[0, 1] / (sA * stau)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.2))

# --- Elipse de covarianza -------------------------------------------------
vals, vecs = np.linalg.eigh(pcov)
ang = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
from matplotlib.patches import Ellipse  # noqa: E402
for k, alpha in [(1, 0.30), (2, 0.15)]:
    ax1.add_patch(Ellipse(popt, 2 * k * np.sqrt(vals[-1]), 2 * k * np.sqrt(vals[0]),
                          angle=ang, color=C.blue, alpha=alpha, lw=0))
ax1.add_patch(Ellipse(popt, 2 * sA, 2 * stau, angle=0, fill=False,
                      edgecolor=C.red, lw=1.6, ls="--"))
ax1.plot(*popt, "o", color=C.ink, ms=6)
ax1.plot(A_REAL, TAU_REAL, "*", color=C.green, ms=13)
ax1.set_xlabel("amplitud $A$"), ax1.set_ylabel(r"tiempo característico $\tau$")
ax1.set_title(f"Elipse de covarianza,  $\\rho$ = {rho:.2f}")
ax1.text(0.03, 0.05, "azul: la incertidumbre real\nrojo: lo que crees si\n"
         "sólo guardas $\\sigma_A$ y $\\sigma_\\tau$",
         transform=ax1.transAxes, fontsize=8.4, color=C.ink)

# --- Consecuencia al predecir --------------------------------------------
tt = np.linspace(0, 12, 200)
M = 4000
muestras_ok = r.multivariate_normal(popt, pcov, M)
muestras_mal = np.column_stack([r.normal(popt[0], sA, M),
                                r.normal(popt[1], stau, M)])
for muestras, color, etiqueta in [(muestras_ok, C.blue, "con covarianza"),
                                  (muestras_mal, C.red, "ignorando $\\rho$")]:
    curvas = np.array([modelo(tt, a, tau) for a, tau in muestras])
    lo, hi = np.percentile(curvas, [2.5, 97.5], axis=0)
    ax2.fill_between(tt, lo, hi, color=color, alpha=0.22, label=etiqueta)
ax2.errorbar(t, y, yerr=SIGMA, fmt="o", color=C.ink, ms=4, lw=1, capsize=2)
ax2.plot(tt, modelo(tt, A_REAL, TAU_REAL), color=C.green, lw=1.6,
         label="verdad")
ax2.set_xlabel("$t$"), ax2.set_ylabel("$y$")
ax2.set_title("Banda de predicción al 95 %")
ax2.legend(fontsize=8)

print(f"A = {popt[0]:.1f} ± {sA:.1f}   tau = {popt[1]:.2f} ± {stau:.2f}   "
      f"rho = {rho:.3f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cuándo miente la fórmula de propagación de errores?

Compara la propagación lineal (derivadas parciales) con Monte Carlo, para una
función suave y para otra fuertemente no lineal.

La figura responde: ¿bajo qué condición la fórmula de la primera derivada da
la respuesta correcta?

Ejecutar:  python fig_propagacion.py

*(script original: `codigo/fig_propagacion.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(8)
N = 400_000

fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.2))

CASOS = [
    # (nombre, f, df, x0, sigma_x, rango)
    (r"$f(x)=x^2$,  $\sigma_x/x_0 = 5\,\%$",
     lambda x: x**2, lambda x: 2 * x, 10.0, 0.5, (60, 145)),
    (r"$f(x)=1/x$,  $\sigma_x/x_0 = 40\,\%$",
     lambda x: 1 / x, lambda x: -1 / x**2, 1.0, 0.4, (0, 6)),
]

for fila, (nombre, f, df, x0, sx, rango) in enumerate(CASOS):
    x = r.normal(x0, sx, N)
    y = f(x)
    sigma_lineal = abs(df(x0)) * sx

    # Panel izquierdo: la función y la anchura de entrada
    ax = axes[fila, 0]
    xx = np.linspace(x0 - 3.2 * sx, x0 + 3.2 * sx, 300)
    xx = xx[xx > 1e-3] if x0 == 1.0 else xx
    ax.plot(xx, f(xx), color=C.blue, lw=2, label="$f(x)$")
    ax.plot(xx, f(x0) + df(x0) * (xx - x0), "--", color=C.ochre, lw=1.6,
            label="aproximación lineal")
    ax.axvspan(x0 - sx, x0 + sx, color=C.red, alpha=0.15)
    ax.set_xlabel("$x$"), ax.set_ylabel("$f(x)$")
    ax.set_title(nombre, fontsize=10)
    ax.legend(fontsize=8)

    # Panel derecho: distribución de salida
    ax = axes[fila, 1]
    ax.hist(y, bins=200, range=rango, density=True, color=C.blue, alpha=0.55,
            edgecolor="none", label="Monte Carlo")
    zz = np.linspace(*rango, 400)
    gauss = np.exp(-0.5 * ((zz - f(x0)) / sigma_lineal) ** 2) / (
        sigma_lineal * np.sqrt(2 * np.pi))
    ax.plot(zz, gauss, color=C.ochre, lw=2, label="propagación lineal")
    ax.axvline(f(x0), color=C.ink, lw=1.4)
    ax.axvline(np.median(y), color=C.red, lw=1.4, ls="--")
    ax.set_xlabel("$f(x)$"), ax.set_ylabel("densidad")
    ax.set_yticks([])
    ax.legend(fontsize=8)
    sesgo = np.mean(y) - f(x0)
    ax.set_title(f"media MC − $f(x_0)$ = {sesgo:+.3g};  "
                 f"$\\sigma_{{MC}}/\\sigma_{{lin}}$ = {y.std()/sigma_lineal:.2f}",
                 fontsize=9.5)
    print(f"{nombre}: sesgo={sesgo:+.4g}  sigma_MC/sigma_lin="
          f"{y.std()/sigma_lineal:.3f}")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Un ajuste con R^2 = 0,998 puede estar completamente mal. ¿Cómo se ve?

Ajusta una recta a datos que en realidad siguen una ley cuadrática suave y
compara con el ajuste correcto. La clave está abajo: los residuos.

La figura responde: ¿qué gráfica detecta un modelo mal especificado, si el
coeficiente de determinación no lo detecta?

Ejecutar:  python fig_residuos.py

*(script original: `codigo/fig_residuos.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(21)

# Datos "experimentales": una curvatura suave más ruido
x = np.linspace(0, 10, 40)
sigma = 0.6
y_real = 2.0 + 1.2 * x + 0.09 * x**2
y = y_real + r.normal(0, sigma, x.size)

ajuste_lineal = np.polyfit(x, y, 1)
ajuste_cuad = np.polyfit(x, y, 2)
res_lin = y - np.polyval(ajuste_lineal, x)
res_cua = y - np.polyval(ajuste_cuad, x)


def r2(res):
    return 1 - np.sum(res**2) / np.sum((y - y.mean()) ** 2)


def chi2_red(res, k):
    return np.sum((res / sigma) ** 2) / (x.size - k)


fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.0), sharex=True,
                         gridspec_kw={"height_ratios": [1.6, 1]})

for col, (nombre, ajuste, res, k) in enumerate([
        ("Modelo lineal", ajuste_lineal, res_lin, 2),
        ("Modelo cuadrático", ajuste_cuad, res_cua, 3)]):
    ax = axes[0, col]
    ax.errorbar(x, y, yerr=sigma, fmt="o", color=C.red, ms=4, lw=1,
                capsize=2, label="datos")
    xx = np.linspace(0, 10, 200)
    ax.plot(xx, np.polyval(ajuste, xx), color=C.blue, lw=2, label="ajuste")
    ax.set_ylabel("$y$")
    ax.set_title(f"{nombre}:  $R^2$ = {r2(res):.4f},  "
                 f"$\\chi^2_\\nu$ = {chi2_red(res, k):.2f}", fontsize=10)
    ax.legend(fontsize=8)

    ax = axes[1, col]
    ax.axhline(0, color=C.ink, lw=1.2)
    ax.axhspan(-sigma, sigma, color=C.grey, alpha=0.18)
    ax.plot(x, res, "o-", color=C.ochre if col == 0 else C.green, ms=4, lw=1)
    ax.set_xlabel("$x$"), ax.set_ylabel("residuo")
    if col == 0:
        ax.annotate("estructura:\nel modelo está mal", xy=(5, res_lin[20]),
                    xytext=(2.2, 1.35), fontsize=8.8, color=C.red,
                    arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
    else:
        ax.text(0.4, 1.35, "sin estructura: sólo ruido", fontsize=8.8,
                color=C.green)
    ax.set_ylim(-1.9, 1.9)

print(f"lineal:     R2={r2(res_lin):.4f}  chi2red={chi2_red(res_lin,2):.2f}")
print(f"cuadrático: R2={r2(res_cua):.4f}  chi2red={chi2_red(res_cua,3):.2f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué promediar mil veces no arregla un error sistemático?

Cuatro dianas con las cuatro combinaciones de sesgo y dispersión, y debajo la
evolución del error de la media con el número de medidas.

La figura responde: ¿qué parte de mi error se reduce midiendo más, y cuál no?

Ejecutar:  python fig_sesgo_dispersion.py

*(script original: `codigo/fig_sesgo_dispersion.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(5)

CASOS = [
    ("Sin sesgo, poca dispersión", 0.0, 0.15, C.green),
    ("Sin sesgo, mucha dispersión", 0.0, 0.55, C.blue),
    ("Con sesgo, poca dispersión", 0.85, 0.15, C.red),
    ("Con sesgo, mucha dispersión", 0.85, 0.55, C.ochre),
]

fig = plt.figure(figsize=(10.4, 6.4))
gs = fig.add_gridspec(2, 4, height_ratios=[1.25, 1.0], hspace=0.42, wspace=0.3)

for i, (titulo, sesgo, disp, color) in enumerate(CASOS):
    ax = fig.add_subplot(gs[0, i])
    for radio in (0.4, 0.8, 1.2):
        ax.add_patch(plt.Circle((0, 0), radio, fill=False, color=C.grey, lw=0.8))
    x = r.normal(sesgo, disp, 60)
    y = r.normal(0.0, disp, 60)
    ax.plot(x, y, "o", color=color, ms=4, alpha=0.75)
    ax.plot(0, 0, "+", color=C.ink, ms=12, mew=1.8)
    ax.set_xlim(-1.6, 1.9), ax.set_ylim(-1.6, 1.6)
    ax.set_aspect("equal"), ax.axis("off")
    ax.set_title(titulo, fontsize=9)

# --- Error de la media frente al número de medidas ------------------------
ax = fig.add_subplot(gs[1, :])
n = np.arange(1, 5001)
for titulo, sesgo, disp, color in CASOS:
    error = np.sqrt(sesgo**2 + (disp / np.sqrt(n)) ** 2)
    ax.loglog(n, error, color=color, lw=1.9, label=titulo)
ax.set_xlabel("número de medidas promediadas $n$")
ax.set_ylabel("error de la media")
ax.set_title("El sesgo es un suelo: no se cruza midiendo más")
ax.legend(fontsize=8.4, ncol=2)
ax.annotate("aquí deja de servir\nseguir midiendo", xy=(400, 0.86),
            xytext=(30, 0.30), fontsize=8.6, color=C.red,
            arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Sale el mismo tiempo característico partiendo de temperaturas distintas?

Tres tazas idénticas que empiezan a 88, 65 y 45 grados, enfriándose en la misma
habitación. Se ajusta la ley de Newton a cada una por separado. La pregunta es
si el tau ajustado depende de la condición inicial: no debería, porque un
sistema lineal de primer orden olvida de dónde viene.

A la derecha, la misma medida en escala logarítmica frente a la temperatura
ambiente ajustada: tres rectas paralelas, y la pendiente comun es -1/tau.

*(script original: `codigo/fig_taza_cafe.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from estilo_libro import C, anota, rng, save, use_style

use_style((9.4, 4.0))

T_AMB = 21.0        # grados, la habitacion
TAU = 24.0          # minutos, propiedad de la taza y del aire
SIGMA = 0.4         # grados, ruido del termometro
INICIALES = [88.0, 65.0, 45.0]

t = np.arange(0.0, 61.0, 2.0)
g = rng(11)


def modelo(t, T_amb, T0, tau):
    return T_amb + (T0 - T_amb) * np.exp(-t / tau)


fig, (ax1, ax2) = plt.subplots(1, 2)
colores = [C.red, C.ochre, C.blue]
ajustes = []

for T0, color in zip(INICIALES, colores):
    T = modelo(t, T_AMB, T0, TAU) + g.normal(0.0, SIGMA, t.size)
    popt, pcov = curve_fit(modelo, t, T, p0=[20.0, T0, 20.0])
    err = np.sqrt(np.diag(pcov))
    ajustes.append((T0, popt, err))
    print(f"T0={T0:5.1f}  ->  T_amb={popt[0]:.2f}  tau={popt[2]:.2f} min"
          f"  (sigma_tau={err[2]:.2f})")

    fino = np.linspace(0.0, 60.0, 400)
    ax1.plot(t, T, "o", color=color, ms=4, alpha=0.75)
    ax1.plot(fino, modelo(fino, *popt), color=color, lw=1.6,
             label=rf"$T_0={T0:.0f}$ °C,  $\tau={popt[2]:.1f}$ min")

    # Escala logaritmica: log(T - T_amb) frente a t es una recta de pendiente -1/tau.
    exceso = T - popt[0]
    valido = exceso > 0.6            # por debajo del ruido el logaritmo miente
    ax2.semilogy(t[valido], exceso[valido], "o", color=color, ms=4, alpha=0.75)
    ax2.semilogy(fino, (popt[1] - popt[0]) * np.exp(-fino / popt[2]),
                 color=color, lw=1.6)

ax1.axhline(T_AMB, color=C.grey, ls="--", lw=1.0, zorder=0)
ax1.text(1.0, T_AMB + 1.2, "temperatura ambiente", ha="left", fontsize=8, color=C.grey)
ax1.set_xlabel("tiempo (min)")
ax1.set_ylabel("temperatura (°C)")
ax1.set_title("Tres tazas, tres puntos de partida")
ax1.legend(loc="upper right")

ax2.set_xlabel("tiempo (min)")
ax2.set_ylabel(r"$T-T_{\mathrm{amb}}$ (°C)")
ax2.set_title("En logaritmo: tres rectas paralelas")
anota(ax2, "misma pendiente = mismo $\\tau$", xy=(30, 12.0), xytext=(4, 3.2),
      color=C.ink)

taus = np.array([p[2] for _, p, _ in ajustes])
print(f"\ntau medio = {taus.mean():.2f} min, dispersion = {taus.std(ddof=1):.2f} min"
      f"  (valor verdadero {TAU:.1f})")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
